# Whisper-large-v2-mn LoRA fine-tune — local RTX 3080 (10 GB)

Adapted from the Colab/T4 notebook. Key changes:
- No Colab / Google Drive — everything runs on local Windows paths
- Loads your **60k-row `data.jsonl`** (`audio_filepath`, `text`, `duration`) directly; log-mel features are computed **on the fly** (precomputing 60k features would need ~55 GB of disk)
- **QLoRA**: base model in 4-bit NF4 (double quant, bf16 compute) + LoRA adapters — ~0.9 GB of weights, tuned for **10 GB VRAM**
- **Checkpoints every 500 steps**, keeps last 3, auto-resumes if you stop/crash and rerun the training cell
- `bf16` instead of `fp16` (3080 is Ampere — more numerically stable)

**One-time setup (in a terminal, before opening this notebook):**
```
pip install torch --index-url https://download.pytorch.org/whl/cu121
pip install -U transformers datasets accelerate evaluate jiwer peft "bitsandbytes>=0.46.1" librosa soundfile tensorboard
```

**Expected training math (~60k rows):** effective batch 16 → ~3,700 optimizer steps per epoch, 2 epochs ≈ 7,400 steps. On a 3080 that's roughly **5–9 h per epoch**. Checkpoints mean you can stop anytime and resume.

## 1. Paths — EDIT the two data paths

In [ ]:
import sys, torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

# Project folder (this notebook, checkpoints, tensorboard logs)
BASE_DIR = Path.home() / "Downloads" / "whisper-project"

# ==== EDIT THESE TWO ====
AUDIO_DIR  = BASE_DIR / "audio"                  # folder containing the audio files
JSONL_PATH = BASE_DIR / "audio" / "data.jsonl"   # the 36k-row jsonl
# ========================

# ==== ONE RUN_ID PER EXPERIMENT ====
# Change this to start a fresh run -> nothing from old runs gets mixed in.
# Keep it the same to resume an interrupted run of THIS experiment.
# Tip: encode the key hyperparams so runs are self-describing.
RUN_ID = "r32-a64-2ep-5e5"
# ===================================

CHECKPOINT_DIR = BASE_DIR / "checkpoints" / RUN_ID                        # resumable checkpoints, per-run
ADAPTER_DIR    = JSONL_PATH.parent / f"mn-whisper-lora-adapter-{RUN_ID}"  # final adapter for this run
LOG_DIR        = BASE_DIR / "tb-logs" / RUN_ID                            # tensorboard logs, per-run

for d in (BASE_DIR, CHECKPOINT_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

MODEL_ID = "bayartsogt/whisper-large-v2-mn-13"

assert JSONL_PATH.exists(), f"data.jsonl not found at {JSONL_PATH} - edit JSONL_PATH above"
assert AUDIO_DIR.exists(),  f"audio folder not found at {AUDIO_DIR} - edit AUDIO_DIR above"
print("Run ID      ->", RUN_ID)
print("Checkpoints ->", CHECKPOINT_DIR)
print("Adapter     ->", ADAPTER_DIR)

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available - reinstall PyTorch with the cu121 index URL above" # is cude available ? is nividia driver working etc
props = torch.cuda.get_device_properties(0)
print(torch.cuda.get_device_name(0), f"| {props.total_memory/1e9:.1f} GB VRAM")

## 2. Processor / tokenizer

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
from tokenizers import AddedToken

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language="Mongolian", task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Mongolian", task="transcribe")


def fix_special_token_flags(tok):
    """This checkpoint ships its Whisper control tokens (<|startoftranscript|>, <|mn|>,
    <|transcribe|>, <|notimestamps|>, ...) with special=False in added_tokens_decoder,
    even though all_special_ids lists them. transformers>=5 decodes through the Rust
    tokenizer, which honours the per-token `special` flag and IGNORES all_special_ids -
    so skip_special_tokens=True silently leaves "<|notimestamps|>" glued to the first
    word and wrecks WER/CER (a perfect transcript scored ~14% WER / ~36% CER).
    Re-register them with special=True. Token ids are unchanged, so existing
    checkpoints and adapters stay valid."""
    tok.add_tokens(
        [AddedToken(t, special=True, normalized=False)
         for t in tok.convert_ids_to_tokens(tok.all_special_ids)],
        special_tokens=True,
    )
    return tok


fix_special_token_flags(tokenizer)
fix_special_token_flags(processor.tokenizer)

# Fail loudly if this ever regresses - a silent failure here corrupts every metric.
_probe = "сайн байна уу!"
_decoded = tokenizer.decode(tokenizer(_probe).input_ids, skip_special_tokens=True)
assert _decoded == _probe, f"special tokens still leaking: {_decoded!r}"
print("special-token flags patched OK ->", repr(_decoded))

## 3. Load the 36k-row jsonl

Resolves audio paths (absolute paths used as-is, otherwise joined with `AUDIO_DIR`), drops rows with missing audio, clips > 30 s (Whisper's input limit), or transcripts > 448 tokens (decoder limit).

In [ ]:
import json, random
from datasets import Dataset

rows, missing, too_long = [], 0, 0
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        p = Path(r["audio_filepath"])
        if not p.is_absolute():
            p = AUDIO_DIR / p
        if not p.exists():
            p2 = AUDIO_DIR / p.name          # last resort: filename only, directly in AUDIO_DIR
            if p2.exists():
                p = p2
            else:
                missing += 1
                continue
        dur = float(r.get("duration", 1.0))
        if not (0.3 <= dur <= 30.0):
            too_long += 1
            continue
        rows.append({"audio_filepath": str(p), "text": r["text"]})

print(f"usable: {len(rows):,} | missing audio: {missing:,} | bad duration: {too_long:,}")

# Drop transcripts longer than Whisper's 448-token decoder limit (~1 min for 60k rows)
n0 = len(rows)
rows = [r for r in rows if len(tokenizer(r["text"]).input_ids) <= 448]
print(f"dropped {n0 - len(rows)} over-long transcripts")

# Train / test split
print("1")
random.Random(42).shuffle(rows)
print("2")

TEST_SIZE = len(rows) // 10
print("3")

test_rows = rows[:TEST_SIZE]
train_rows = rows[TEST_SIZE:]
print("4")

train_ds = Dataset.from_list(train_rows)
print("5")

test_ds = Dataset.from_list(test_rows)          # FULL held-out set (~10%), evaluated once at the very end

# Small FIXED subset for fast in-training WER/CER. predict_with_generate makes eval slow
# (autoregressive decode per sample), so we DON'T generate over the whole test set every eval_steps.
# rows were already shuffled with seed=42, so the first EVAL_SUBSET rows are a representative sample.
EVAL_SUBSET = 500
eval_small = test_ds.select(range(min(EVAL_SUBSET, len(test_ds))))

print(train_ds, test_ds)
print(f"eval_small (in-training): {len(eval_small):,} rows")
print("6")

## 4. On-the-fly feature extraction

Audio is loaded and converted to log-mel features at batch time — nothing is precomputed to disk.

In [ ]:
import librosa

def prepare(batch):
    feats, labs = [], []
    for path, text in zip(batch["audio_filepath"], batch["text"]):
        audio, _ = librosa.load(path, sr=16000)
        feats.append(feature_extractor(audio, sampling_rate=16000).input_features[0])
        labs.append(tokenizer(text).input_ids)
    return {"input_features": feats, "labels": labs}

for ds in (train_ds, test_ds, eval_small):
    ds.set_transform(prepare)

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# The tokenizer prepends <|startoftranscript|> to every label sequence, and the model
# ALSO prepends decoder_start_token_id when it builds decoder_input_ids from labels.
# So the label's leading <|startoftranscript|> has to be stripped, or it ends up doubled
# and the model trains on a prefix that never occurs at generation time.
#
# NOTE: the usual `labels[:, 0] == tokenizer.bos_token_id` check is WRONG for Whisper -
# bos_token_id here is 50257 (<|endoftext|>) while the label starts with 50258
# (<|startoftranscript|>), so the strip silently never fired.
DECODER_START_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
assert DECODER_START_TOKEN_ID != tokenizer.unk_token_id, "could not resolve <|startoftranscript|>"

# Do NOT try to -100 the <|mn|>/<|transcribe|>/<|notimestamps|> prefix out of `labels`:
# the model derives decoder_input_ids from labels via shift_tokens_right, which turns
# -100 into pad_token_id (= <|endoftext|>). Masking them destroys the decoder's context
# prefix, which is far worse than leaving them in. They stay - see the inspect cell below.


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # -100 marks positions the loss must ignore (padding only)
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Drop the leading <|startoftranscript|>; the model re-adds it via shift_tokens_right.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=DECODER_START_TOKEN_ID,
)

# Verify the strip actually fires now (it silently did not before).
_probe = [{"input_features": [[0.0] * 3000] * 80, "labels": tokenizer("сайн уу").input_ids}]
_batch = data_collator(_probe)
assert _batch["labels"][0, 0].item() != DECODER_START_TOKEN_ID, "leading <|sot|> was NOT stripped"
print("collator OK - first label token is now:",
      tokenizer.convert_ids_to_tokens(_batch["labels"][0, 0].item()))

## 5. Model: QLoRA — 4-bit NF4 base + LoRA (fits easily in 10 GB)

In [ ]:
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
from transformers.models.whisper.tokenization_whisper import LANGUAGES
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# QLoRA: 4-bit NF4 quantization, double quant, bf16 compute
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization_config, device_map="auto"
)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "mn"
model.generation_config.task = "transcribe"

# This checkpoint ships no generation_config.json, so lang_to_id is missing -
# rebuild it from the tokenizer's language tokens (the ones present in its vocab)
model.generation_config.lang_to_id = {
    f"<|{lang}|>": tid
    for lang in LANGUAGES
    if (tid := tokenizer.convert_tokens_to_ids(f"<|{lang}|>")) != tokenizer.unk_token_id
}

model.generation_config.task_to_id = {
    task: tid
    for task in ("transcribe", "translate")
    if (tid := tokenizer.convert_tokens_to_ids(f"<|{task}|>")) != tokenizer.unk_token_id
}

model = prepare_model_for_kbit_training(model)   # also enables gradient checkpointing
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import evaluate

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

## 6. Training arguments — tuned for 3080 / 10 GB

Checkpoint plan: **save every 500 steps** (~every 2 h), keep the **last 3**, quick WER/CER on 150 samples **every 1000 steps**. If you hit CUDA OOM, set `PER_DEVICE_BS = 4` and `GRAD_ACCUM = 4` (same effective batch).

In [ ]:
import math
from transformers import Seq2SeqTrainingArguments

PER_DEVICE_BS = 4    # 4-bit QLoRA frees VRAM vs int8 -> 8 fits in 10 GB
GRAD_ACCUM    = 2    # effective batch = 8
NUM_EPOCHS    = 3    # ~87k rows -> ~8100 steps total; raise to 3 if WER is still improving
# 4350 
steps_per_epoch = math.ceil(len(train_ds) / (PER_DEVICE_BS * GRAD_ACCUM))
max_steps = steps_per_epoch * NUM_EPOCHS
print(f"steps/epoch: {steps_per_epoch:,} | total steps: {max_steps:,}")

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=4,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    lr_scheduler_type="linear",

    warmup_steps=980,               # ~3% of total steps
    max_steps=max_steps,
    seed=42,
    bf16=True,                      # Ampere (3080): bf16 > fp16 for stability
    eval_strategy="steps",
    eval_steps=1000,                 # quick WER/CER on eval subset
    save_strategy="steps",
    save_steps=1000,                 # MUST be a multiple of eval_steps when load_best_model_at_end=True
    save_total_limit=3,             # keep last 3 (the best checkpoint is always protected on top)
    load_best_model_at_end=True,    # reload the lowest-WER checkpoint when training ends
    metric_for_best_model="wer",    # pick "best" by WER (Trainer looks at eval_wer)...
    greater_is_better=False,        # ...lower WER is better
    logging_steps=50,
    logging_dir=str(LOG_DIR),
    predict_with_generate=True,
    generation_max_length=225,
    report_to=["tensorboard"],
    remove_unused_columns=False,    # required for PEFT
    label_names=["labels"],
    dataloader_num_workers=0,       # keep 0 on Windows (set_transform + worker processes don't mix)
    dataloader_pin_memory=True,
)

## 7. Train (auto-resumes from latest checkpoint)

Stop anytime (interrupt kernel / power loss) — rerunning this cell picks up from the newest checkpoint in `CHECKPOINT_DIR`. Monitor with: `tensorboard --logdir <BASE_DIR>\tb-logs`

In [ ]:
import os, re
from transformers import Seq2SeqTrainer, TrainerCallback, EarlyStoppingCallback


class SavePeftModelCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint_folder = f"{args.output_dir}/checkpoint-{state.global_step}"
        kwargs["model"].save_pretrained(checkpoint_folder)
        return control


trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_small,        # small subset during training; full test set evaluated after
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        SavePeftModelCallback,
        # Stop if WER hasn't improved for 3 consecutive evals; best model is reloaded at the end.
        EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0),
    ],
)

model.config.use_cache = False   # required with gradient checkpointing

# Resume from the latest checkpoint in THIS run's folder only (RUN_ID keeps runs isolated).
# Match ONLY directories named exactly "checkpoint-<int>" - a stray "checkpoint-9000.zip"
# (or any other non-checkpoint file) used to crash int() here.
CKPT_RE = re.compile(r"^checkpoint-(\d+)$")

candidates = []
for d in os.listdir(CHECKPOINT_DIR):
    m = CKPT_RE.match(d)
    if m and (CHECKPOINT_DIR / d).is_dir() and (CHECKPOINT_DIR / d / "trainer_state.json").exists():
        candidates.append((int(m.group(1)), d))

last_checkpoint = None
if candidates:
    last_checkpoint = str(CHECKPOINT_DIR / max(candidates)[1])
    print(f"Resuming from {last_checkpoint}")
else:
    print("No usable checkpoint found - starting from scratch")

trainer.train(resume_from_checkpoint=last_checkpoint)


# Final evaluation on the full held-out test set, right after training finishes.
# Thanks to load_best_model_at_end=True, this runs on the lowest-WER adapter, not the last step.
eval_metrics = trainer.evaluate(eval_dataset=test_ds)
print(eval_metrics)

## 8. Save the adapter (next to data.jsonl)

In [ ]:
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

## 9. Quick sanity check — transcribe one test clip

In [ ]:
sample = test_rows[0]
audio, _ = librosa.load(sample["audio_filepath"], sr=16000)
inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)

model.eval()
model.config.use_cache = True
with torch.no_grad(), torch.autocast("cuda"):
    ids = model.generate(inputs, language="mn", task="transcribe", max_new_tokens=225)
    

print("REF:", sample["text"])
print("HYP:", tokenizer.batch_decode(ids, skip_special_tokens=True)[0])